# Ch.3 — Backprop & Optimisers
**Track:** ML from Scratch · California Housing dataset

## Core Idea

**Backpropagation** applies the chain rule backwards through the network to compute $\nabla_W \mathcal{L}$ for every weight.

**Optimisers** use those gradients to update weights:
```
SGD  →  Momentum  →  RMSProp  →  Adam
(bare)  (velocity)   (adaptive scale)  (both combined)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error

housing = fetch_california_housing()
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape}  Test: {X_test_s.shape}")

## Manual Backprop (NumPy)

Network: 8 → 64 → 32 → 1  
Loss: MSE. One forward pass + backward pass from scratch.

Forward pass stores:
- $z^{(l)}$ — pre-activation (needed for ReLU derivative)
- $h^{(l)}$ — post-activation (needed for weight gradient)

Backward pass:
$$\delta^{(L)} = \frac{2}{n}(\hat{y} - y) \qquad \delta^{(l)} = (W_{l+1}\,\delta^{(l+1)}) \odot \mathbf{1}[z^{(l)}>0]$$
$$\nabla_{W_l} = h^{(l-1)\top} \delta^{(l)} \qquad \nabla_{b_l} = \sum_\text{batch} \delta^{(l)}$$

In [ ]:
# ── He initialisation + manual forward/backward pass ──────
rng = np.random.default_rng(42)
ARCH = [8, 64, 32, 1]

def he_init(n_in, n_out):
    return rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out))

# Initialise weights and biases for all layers
params = []
for n_in, n_out in zip(ARCH[:-1], ARCH[1:]):
    params.append({'W': he_init(n_in, n_out), 'b': np.zeros(n_out)})

def relu(z):          return np.maximum(0, z)
def relu_grad(z):     return (z > 0).astype(float)   # 1 where z>0, else 0

def forward(X, params):
    """Forward pass. Returns (y_hat, cache).
    cache: list of (z, h) per layer, where h is passed to the next layer.
    """
    cache = []
    h = X
    for i, p in enumerate(params):
        z = h @ p['W'] + p['b']
        # Last layer: linear output (no activation)
        h_next = z if i == len(params) - 1 else relu(z)
        cache.append({'z': z, 'h_in': h})
        h = h_next
    return h.ravel(), cache

def backward(y_hat, y, cache, params):
    """Backward pass. Returns list of {dW, db} per layer."""
    n = len(y)
    grads = [None] * len(params)

    # Output layer error (MSE derivative: 2/n * (ŷ - y))
    delta = (2 / n) * (y_hat - y).reshape(-1, 1)   # (n, 1)

    for l in range(len(params) - 1, -1, -1):
        z     = cache[l]['z']
        h_in  = cache[l]['h_in']

        # Weight and bias gradients
        dW = h_in.T @ delta                  # (n_in, n_out)
        db = delta.sum(axis=0)               # (n_out,)
        grads[l] = {'dW': dW, 'db': db}

        if l > 0:  # propagate error to layer below
            delta = (delta @ params[l]['W'].T) * relu_grad(cache[l-1]['z'])

    return grads

# One forward + backward pass
y_hat, cache = forward(X_train_s[:64], params)
grads = backward(y_hat, y_train[:64], cache, params)

print("Layer | Weight gradient shape | Max |grad|")
print("-" * 50)
for i, g in enumerate(grads):
    print(f"  {i+1}   | {str(g['dW'].shape):22s} | {np.abs(g['dW']).max():.5f}")

## Manual SGD Training Loop

Train the network for 200 epochs with vanilla SGD. Store the loss at each epoch so we can compare with other optimisers.

In [ ]:
def mse_loss(y_hat, y):
    return np.mean((y_hat - y) ** 2)

def train_sgd(X, y, lr=0.01, epochs=200, batch_size=256, seed=42):
    rng = np.random.default_rng(seed)
    local_params = [
        {'W': he_init(n_in, n_out), 'b': np.zeros(n_out)}
        for n_in, n_out in zip(ARCH[:-1], ARCH[1:])
    ]
    losses = []
    n = len(X)
    for epoch in range(epochs):
        # Shuffle
        idx = rng.permutation(n)
        epoch_loss = 0.0
        for start in range(0, n, batch_size):
            Xb = X[idx[start:start+batch_size]]
            yb = y[idx[start:start+batch_size]]
            y_hat, cache = forward(Xb, local_params)
            epoch_loss += mse_loss(y_hat, yb) * len(Xb)
            grads = backward(y_hat, yb, cache, local_params)
            for p, g in zip(local_params, grads):
                p['W'] -= lr * g['dW']
                p['b'] -= lr * g['db']
        losses.append(epoch_loss / n)
    return local_params, losses

print("Training with SGD (lr=0.01)...")
_, sgd_losses = train_sgd(X_train_s, y_train, lr=0.01, epochs=150)
print(f"Final loss: {sgd_losses[-1]:.4f}")

## Optimiser Comparison — sklearn MLPRegressor

sklearn exposes SGD, SGD+Momentum, and Adam via `solver` and `momentum` params. We compare loss curves and final R².

In [ ]:
EPOCHS = 300
ARCH_SK = (128, 64)   # sklearn uses the same 2-hidden-layer architecture from Ch.2

configs = [
    ('SGD',           dict(solver='sgd',  learning_rate_init=0.01, momentum=0.0)),
    ('SGD+Momentum',  dict(solver='sgd',  learning_rate_init=0.01, momentum=0.9)),
    ('Adam',          dict(solver='adam', learning_rate_init=1e-3)),
]

results = {}
for name, kwargs in configs:
    m = MLPRegressor(
        hidden_layer_sizes=ARCH_SK,
        activation='relu',
        max_iter=EPOCHS,
        random_state=42,
        **kwargs
    )
    m.fit(X_train_s, y_train)
    r2 = r2_score(y_test, m.predict(X_test_s))
    results[name] = {'model': m, 'r2': r2, 'loss': m.loss_curve_}
    print(f"{name:<18}  R²={r2:.4f}  epochs_run={m.n_iter_}")

In [ ]:
# Loss curves side by side + R² bar chart
colors = {'SGD': 'steelblue', 'SGD+Momentum': 'darkorange', 'Adam': 'green'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss curves
for name, res in results.items():
    axes[0].plot(res['loss'], label=name, color=colors[name], alpha=0.85)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training MSE loss')
axes[0].set_title('Optimiser Loss Curves')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(alpha=0.3)

# R² bar chart
names = list(results.keys())
r2s   = [results[n]['r2'] for n in names]
bars  = axes[1].bar(names, r2s, color=[colors[n] for n in names], alpha=0.8)
axes[1].set_ylabel('R² (test set)')
axes[1].set_title('Final R² by Optimiser')
axes[1].set_ylim(0, 1)
axes[1].grid(axis='y', alpha=0.3)
for bar, r2 in zip(bars, r2s):
    axes[1].text(bar.get_x() + bar.get_width()/2, r2 + 0.01, f'{r2:.3f}',
                 ha='center', va='bottom', fontsize=10)

plt.suptitle('SGD vs Momentum vs Adam on California Housing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Manual Adam Implementation (NumPy)

Implement Adam from scratch to see the bias-correction mechanism in action.

In [ ]:
class AdamOptimiser:
    """Adam optimiser with bias correction."""
    def __init__(self, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr; self.beta1 = beta1; self.beta2 = beta2; self.eps = eps
        self.m = {}; self.v = {}; self.t = 0

    def step(self, params, grads):
        self.t += 1
        for i, (p, g) in enumerate(zip(params, grads)):
            for key in ('W', 'b'):
                k = (i, key)
                grad = g['d' + key]
                # First moment (velocity)
                self.m[k] = self.beta1 * self.m.get(k, np.zeros_like(grad)) + (1 - self.beta1) * grad
                # Second moment (variance)
                self.v[k] = self.beta2 * self.v.get(k, np.zeros_like(grad)) + (1 - self.beta2) * grad**2
                # Bias correction
                m_hat = self.m[k] / (1 - self.beta1 ** self.t)
                v_hat = self.v[k] / (1 - self.beta2 ** self.t)
                # Weight update
                p[key] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

def train_adam(X, y, lr=1e-3, epochs=200, batch_size=256, seed=42):
    rng = np.random.default_rng(seed)
    local_params = [
        {'W': he_init(n_in, n_out), 'b': np.zeros(n_out)}
        for n_in, n_out in zip(ARCH[:-1], ARCH[1:])
    ]
    opt = AdamOptimiser(lr=lr)
    losses = []
    n = len(X)
    for epoch in range(epochs):
        idx = rng.permutation(n)
        epoch_loss = 0.0
        for start in range(0, n, batch_size):
            Xb = X[idx[start:start+batch_size]]
            yb = y[idx[start:start+batch_size]]
            y_hat, cache = forward(Xb, local_params)
            epoch_loss += mse_loss(y_hat, yb) * len(Xb)
            grads = backward(y_hat, yb, cache, local_params)
            opt.step(local_params, grads)
        losses.append(epoch_loss / n)
    return local_params, losses

print("Training with manual Adam (lr=0.001)...")
adam_params, adam_losses = train_adam(X_train_s, y_train, lr=1e-3, epochs=150)

# Compare SGD vs Adam (manual) loss curves
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sgd_losses,  label='Manual SGD  (lr=0.01)',   color='steelblue')
ax.plot(adam_losses, label='Manual Adam (lr=0.001)',   color='green')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Manual SGD vs Manual Adam — Loss Curves')
ax.legend(); ax.grid(alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

# Evaluate manual Adam on test set
y_hat_adam, _ = forward(X_test_s, adam_params)
print(f"Manual Adam R² on test: {r2_score(y_test, y_hat_adam):.4f}")

## Learning Rate Sweep

What happens when learning rate is too high, too low, or just right?

In [ ]:
lr_values = [1e-4, 1e-3, 1e-2, 0.1]
lr_results = {}

for lr in lr_values:
    m = MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        learning_rate_init=lr,
        max_iter=200,
        random_state=42,
    )
    m.fit(X_train_s, y_train)
    r2 = r2_score(y_test, m.predict(X_test_s))
    lr_results[lr] = {'loss': m.loss_curve_, 'r2': r2, 'epochs': m.n_iter_}
    print(f"lr={lr:.0e}  R²={r2:.4f}  epochs_run={m.n_iter_}")

fig, ax = plt.subplots(figsize=(10, 4))
cmap = plt.colormaps['viridis']
for i, (lr, res) in enumerate(lr_results.items()):
    ax.plot(res['loss'], label=f'lr={lr:.0e}  R²={res["r2"]:.3f}',
            color=cmap(i / len(lr_results)))
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Learning Rate Sweep (Adam, 128→64→1)')
ax.legend(); ax.grid(alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## Trap 1 — High Learning Rate Divergence

A learning rate that is too large causes the optimizer to step over the minimum. Loss oscillates or diverges entirely.

In [ ]:
# Show divergence with a very large LR using the manual SGD loop
# We handle the NaN case gracefully to avoid crashing the notebook

def train_sgd_safe(X, y, lr=1.0, epochs=40, batch_size=256, seed=42):
    """Returns losses; stops early if NaN is detected."""
    rng = np.random.default_rng(seed)
    local_params = [
        {'W': he_init(n_in, n_out), 'b': np.zeros(n_out)}
        for n_in, n_out in zip(ARCH[:-1], ARCH[1:])
    ]
    losses = []
    n = len(X)
    for epoch in range(epochs):
        idx = rng.permutation(n)
        epoch_loss = 0.0
        for start in range(0, n, batch_size):
            Xb = X[idx[start:start+batch_size]]
            yb = y[idx[start:start+batch_size]]
            y_hat, cache = forward(Xb, local_params)
            loss_val = mse_loss(y_hat, yb)
            if np.isnan(loss_val) or loss_val > 1e8:
                losses.append(float('nan'))
                return losses   # stop early — diverged
            epoch_loss += loss_val * len(Xb)
            grads = backward(y_hat, yb, cache, local_params)
            for p, g in zip(local_params, grads):
                p['W'] -= lr * g['dW']
                p['b'] -= lr * g['db']
        losses.append(epoch_loss / n)
    return losses

losses_good = train_sgd_safe(X_train_s, y_train, lr=0.01,  epochs=60)
losses_bad  = train_sgd_safe(X_train_s, y_train, lr=5.0,   epochs=60)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(losses_good, label='lr=0.01  (converges)',  color='steelblue')

# Replace NaN with a large sentinel for visualisation
vis_bad = [l if (l and not np.isnan(l)) else 1e5 for l in losses_bad]
ax.plot(vis_bad, label='lr=5.0  (diverges / NaN)', color='red', linestyle='--')

ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Trap: Learning Rate Too High → Divergence')
ax.legend(); ax.grid(alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()
print(f"Good LR — final loss: {losses_good[-1]:.4f}")
print(f"Bad LR  — diverged at epoch {len(losses_bad)} (NaN or > 1e8)")

## Trap 2 — Adam Masks Bad Architecture

Adam's adaptive scaling can make even a poor architecture look like it converges. Switch to SGD to expose the real quality of the model.

In [ ]:
# Compare a tiny (underpowered) network with Adam vs SGD
# Adam makes it look OK; SGD reveals its true limitations

tiny = (4,)   # deliberately underpowered: 1 hidden layer of 4 units
good = (128, 64)

rows = []
for arch, arch_name in [(tiny, 'Tiny (4)'), (good, 'Good (128,64)')]:
    for solver, lr in [('adam', 1e-3), ('sgd', 0.01)]:
        m = MLPRegressor(
            hidden_layer_sizes=arch, activation='relu',
            solver=solver, learning_rate_init=lr,
            max_iter=300, random_state=42
        ).fit(X_train_s, y_train)
        r2_tr = r2_score(y_train, m.predict(X_train_s))
        r2_te = r2_score(y_test,  m.predict(X_test_s))
        rows.append((arch_name, solver.upper(), f'{r2_tr:.3f}', f'{r2_te:.3f}'))

print(f"{'Architecture':<16} {'Solver':<8} {'R²(train)':>10} {'R²(test)':>10}")
print("-" * 50)
for row in rows:
    print(f"{row[0]:<16} {row[1]:<8} {row[2]:>10} {row[3]:>10}")

print("\n→ Tiny network with Adam may show acceptable train R²,")
print("  but SGD reveals the same architecture is actually underpowered.")

## Exercises

**Exercise 1 — Gradient clipping**  
Modify `train_sgd_safe` to clip gradients to $[-1, 1]$ before the weight update (`np.clip`). Does it allow a higher learning rate (e.g., 0.5) to converge without diverging?

**Exercise 2 — Cosine LR schedule**  
Implement a cosine annealing schedule:  
$\eta_t = \eta_{\min} + \tfrac{1}{2}(\eta_{\max}-\eta_{\min})(1 + \cos(\pi t / T))$  
Update the manual Adam loop to use it. Plot the LR schedule alongside the loss curve.

**Exercise 3 — Momentum visualisation**  
Using `MLPRegressor(solver='sgd', momentum=m)` for `m` in `[0.0, 0.5, 0.9, 0.99]`, plot the 4 loss curves on a single plot. At what momentum value does the optimiser start overshooting?

In [ ]:
# Exercise 1 scaffold — gradient clipping
def train_sgd_clipped(X, y, lr=0.5, clip=1.0, epochs=60, batch_size=256, seed=42):
    rng = np.random.default_rng(seed)
    local_params = [
        {'W': he_init(n_in, n_out), 'b': np.zeros(n_out)}
        for n_in, n_out in zip(ARCH[:-1], ARCH[1:])
    ]
    losses = []
    n = len(X)
    for epoch in range(epochs):
        idx = rng.permutation(n)
        epoch_loss = 0.0
        for start in range(0, n, batch_size):
            Xb = X[idx[start:start+batch_size]]
            yb = y[idx[start:start+batch_size]]
            y_hat, cache = forward(Xb, local_params)
            epoch_loss += mse_loss(y_hat, yb) * len(Xb)
            grads = backward(y_hat, yb, cache, local_params)
            for p, g in zip(local_params, grads):
                # TODO: clip gradients here using np.clip before updating
                p['W'] -= lr * g['dW']
                p['b'] -= lr * g['db']
        losses.append(epoch_loss / n)
    return losses

# losses_clipped = train_sgd_clipped(X_train_s, y_train, lr=0.5, clip=1.0)
# Compare with losses_bad (lr=5.0 diverged) and plot

In [ ]:
# Exercise 2 scaffold — cosine LR schedule
def cosine_lr(epoch, T, lr_min=1e-5, lr_max=1e-3):
    """Cosine annealing schedule."""
    # TODO: return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * epoch / T))
    return lr_max  # placeholder — replace with formula

# Modify train_adam to accept a schedule function and call cosine_lr(epoch, T=150)
# Plot LR schedule on left axis, loss on right axis

In [ ]:
# Exercise 3 scaffold — momentum sweep
momentum_values = [0.0, 0.5, 0.9, 0.99]

fig, ax = plt.subplots(figsize=(10, 4))
for mu in momentum_values:
    m_model = MLPRegressor(
        hidden_layer_sizes=(128, 64), activation='relu',
        solver='sgd', learning_rate_init=0.01, momentum=mu,
        max_iter=200, random_state=42
    ).fit(X_train_s, y_train)
    ax.plot(m_model.loss_curve_, label=f'momentum={mu}')

ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Momentum Sweep — Loss Curves')
ax.legend(); ax.grid(alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()
# TODO: observe at which momentum value oscillations appear